In [93]:
"""
Builds two district crosswalks for the NFHS panel:

(1) NFHS-1 / NFHS-2 -> 1991 Census district, via fuzzy name matching within
    state. Falls back to the 1981 Census sheet for states with no 1991 entry
    (e.g. Jammu & Kashmir).

(2) NFHS-4 / NFHS-5 -> stable-district baseline, via direct code join on
    each round's own district variable (sdistri / sdist). Restricted to the
    575 districts with unchanged codes across both rounds, per DHS guidance
    (https://userforum.dhsprogram.com/index.php?t=msg&goto=24551) -- no
    attempt is made to reconstruct the 65 split parent districts, since no
    official parent->child concordance exists. One code-numbering swap
    (Sangrur / Shahid Bhagat Singh Nagar, Punjab) is corrected via
    CODE_SWAP_NFHS5_TO_NFHS4, confirmed against the GPS reference files.

INPUT FILES REQUIRED:
    data/raw/DHS_Districts/NFHS-1_DistrictCodes.xlsx
    data/raw/DHS_Districts/NFHS-2_DistrictCodes.xlsx
    data/raw/DHS_Districts/dist_list_81_91.xlsx
    data/raw/DHS_Districts/IAGE71FL.dbf   (NFHS-4 GPS reference)
    data/raw/DHS_Districts/IAGE7AFL.dbf   (NFHS-5 GPS reference)
    data/raw/DHS_microdata/NFHS4_2015-16_IndividualRecode/IAIR74FL.DTA
    data/raw/DHS_microdata/NFHS5_2019-21_IndividualRecode/IAIR7EFL.DTA

OUTPUT (data/processed/district_crosswalks/):
    nfhs1_district_crosswalk.csv
    nfhs2_district_crosswalk.csv
    crosswalk_review_flags.csv               (NFHS-1/2 matches below MATCH_THRESHOLD)
    nfhs4_nfhs5_stable_district_crosswalk.csv
    nfhs4_nfhs5_excluded_split_districts.csv
    nfhs4_gps_district_reference.csv
    nfhs5_gps_district_reference.csv

STATUS (as of last full run):
    NFHS-1/2: 393/440 rows, all NE-state overrides applied, remaining
        duplicate district_uids (Delhi, Tripura, Goa, Sikkim) are confirmed
        legitimate 1991-boundary collapses, not errors.
    NFHS-4/5: 575 stable districts, 65 excluded (documented split parents),
        10 cosmetic name-spelling mismatches (harmless), 1 code swap fixed.
"""

"\nBuilds two district crosswalks for the NFHS panel:\n\n(1) NFHS-1 / NFHS-2 -> 1991 Census district, via fuzzy name matching within\n    state. Falls back to the 1981 Census sheet for states with no 1991 entry\n    (e.g. Jammu & Kashmir).\n\n(2) NFHS-4 / NFHS-5 -> stable-district baseline, via direct code join on\n    each round's own district variable (sdistri / sdist). Restricted to the\n    575 districts with unchanged codes across both rounds, per DHS guidance\n    (https://userforum.dhsprogram.com/index.php?t=msg&goto=24551) -- no\n    attempt is made to reconstruct the 65 split parent districts, since no\n    official parent->child concordance exists. One code-numbering swap\n    (Sangrur / Shahid Bhagat Singh Nagar, Punjab) is corrected via\n    CODE_SWAP_NFHS5_TO_NFHS4, confirmed against the GPS reference files.\n\nINPUT FILES REQUIRED:\n    data/raw/DHS_Districts/NFHS-1_DistrictCodes.xlsx\n    data/raw/DHS_Districts/NFHS-2_DistrictCodes.xlsx\n    data/raw/DHS_Districts/dist_l

In [94]:
import pandas as pd
import geopandas as gpd
from rapidfuzz import process, fuzz
import re
import pyreadstat
from pathlib import Path
from manual_overrides import MANUAL_OVERRIDES

# Resolve repo root whether the notebook cwd is scripts/ or the project root
_cwd = Path.cwd()
if (_cwd / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise FileNotFoundError(
        "Could not find data/raw/DHS_Districts from cwd "
        f"{_cwd}. Run from the repo root or scripts/."
    )

RAW_DIR = REPO_ROOT / "data" / "raw" / "DHS_Districts"
OUT_DIR = REPO_ROOT / "data" / "processed" / "district_crosswalks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MATCH_THRESHOLD = 90  # below this score, flag for manual review
print(f"RAW_DIR={RAW_DIR}")
print(f"OUT_DIR={OUT_DIR}")

RAW_DIR=/Users/eknoorsandhu/DHS_India_Research/data/raw/DHS_Districts
OUT_DIR=/Users/eknoorsandhu/DHS_India_Research/data/processed/district_crosswalks


In [95]:
# ---------------------------------------------------------------------------
# State name normalization: maps the messy variants in each source file to
# one canonical key so we can join state-by-state. Extend this if you hit
# a state that doesn't resolve (the script will print unmatched states).
# ---------------------------------------------------------------------------
STATE_ALIASES = {
    "andhra pradesh": "andhra pradesh",
    "arunachalpradesh": "arunachal pradesh",
    "arunachal pradesh": "arunachal pradesh",
    "assam": "assam",
    "bihar": "bihar",
    "goa": "goa",
    "gujarat": "gujarat",
    "haryana": "haryana",
    "himachal pradesh": "himachal pradesh",
    "himachal": "himachal pradesh",
    "jammu": "jammu & kashmir",
    "jammu & kashmir": "jammu & kashmir",
    "j & k": "jammu & kashmir",
    "karnataka": "karnataka",
    "kerala": "kerala",
    "madhya pradesh": "madhya pradesh",
    "maharashtra": "maharashtra",
    "manipur": "manipur",
    "meghalaya": "meghalaya",
    "mizoram": "mizoram",
    "nagaland": "nagaland",
    "new delhi": "delhi",
    "delhi": "delhi",
    "orissa": "orissa",
    "punjab": "punjab",
    "rajasthan": "rajasthan",
    "sikkim": "sikkim",
    "tamil nadu": "tamil nadu",
    "tripura": "tripura",
    "uttar pradesh": "uttar pradesh",
    "west bengal": "west bengal",
    # Census-only union territories / abbreviated 1981 labels -- these never
    # appear in the NFHS-1/2 state lists, kept here only to silence warnings
    # and allow the 1981-sheet fallback groupby to run cleanly.
    "andaman & nicobar": "andaman & nicobar",
    "andaman": "andaman & nicobar",
    "chandigarh": "chandigarh",
    "dadra + nagar haveli": "dadra & nagar haveli",
    "dadra +": "dadra & nagar haveli",
    "daman & diu": "daman & diu",
    "laksdadweep": "lakshadweep",
    "laccadive +": "lakshadweep",
    "pondicherry": "pondicherry",
    "andhra": "andhra pradesh",
    "arunachal": "arunachal pradesh",
    "goa +": "goa",
}

In [96]:
def norm_state(s):
    key = str(s).strip().lower()
    if key not in STATE_ALIASES:
        print(f"  [WARN] Unrecognized state name '{s}' -- add to STATE_ALIASES")
        return key
    return STATE_ALIASES[key]
 
 
def norm_district(s):
    """Lowercase, strip parentheticals and extra whitespace for fuzzy matching."""
    s = str(s).strip().lower()
    s = re.sub(r"\(.*?\)", "", s)      # drop "(ongole)" style parentheticals
    s = re.sub(r"[^a-z\s]", "", s)      # drop punctuation/ampersands
    s = re.sub(r"\s+", " ", s).strip()
    return s
 
 
def build_census_lookup(census91_df, census81_df):
    """
    Returns dict: canonical_state -> list of (raw_dlabel, normalized_dlabel, distid)
    Falls back to 1981 sheet for states absent from the 1991 sheet.
    """
    lookup = {}
    for df, tag in [(census91_df, "1991"), (census81_df, "1981")]:
        for state_raw, group in df.groupby("slabel"):
            state = norm_state(state_raw)
            if state in lookup:
                continue  # already have 1991 version, skip 1981 fallback
            lookup[state] = {
                "source_year": tag,
                "districts": [
                    (row["dlabel"], norm_district(row["dlabel"]), row["distid"])
                    for _, row in group.iterrows()
                ],
            }
    return lookup

In [97]:
def match_round(nfhs_df, census_lookup, round_label):
    """
    For each (state, shdist) in the NFHS crosswalk file, fuzzy-match the
    district name against the Census district list for that state.
    """
    results = []
    flags = []
 
    nfhs_districts = (
        nfhs_df[["hv024", "state_name", "shdist", "District"]]
        .drop_duplicates()
        .copy()
    )
 
    for _, row in nfhs_districts.iterrows():
        state = norm_state(row["state_name"])
        nfhs_dist_raw = row["District"]

        # Skip blank/null district names -- these have no lexical content to
        # fuzzy-match against, and were previously getting silently assigned
        # to whatever the scorer's least-bad guess happened to be (often
        # scoring >= MATCH_THRESHOLD and so slipping past the flag check).
        if pd.isna(nfhs_dist_raw) or str(nfhs_dist_raw).strip() == "":
            results.append({
                "round": round_label,
                "hv024": row["hv024"],
                "state": state,
                "shdist": row["shdist"],
                "nfhs_district_name": nfhs_dist_raw,
                "matched_census_district": None,
                "census_distid": None,
                "census_source_year": None,
                "match_score": 0,
                "match_source": "unresolved_blank_district_name",
            })
            flags.append({**results[-1], "reason": "blank/null district name in source file"})
            continue

        nfhs_dist_norm = norm_district(nfhs_dist_raw)
 
        census_entry = census_lookup.get(state)
        if census_entry is None:
            results.append({
                "round": round_label,
                "hv024": row["hv024"],
                "state": state,
                "shdist": row["shdist"],
                "nfhs_district_name": nfhs_dist_raw,
                "matched_census_district": None,
                "census_distid": None,
                "census_source_year": None,
                "match_score": 0,
            })
            flags.append({**results[-1], "reason": "no census entry for state"})
            continue
 
        choices = [d[1] for d in census_entry["districts"]]
        best = process.extractOne(nfhs_dist_norm, choices, scorer=fuzz.WRatio)
 
        if best is None:
            match_name, score, idx = None, 0, None
        else:
            match_name, score, idx = best
 
        if idx is not None:
            matched_raw, _, matched_distid = census_entry["districts"][idx]
        else:
            matched_raw, matched_distid = None, None
 
        record = {
            "round": round_label,
            "hv024": row["hv024"],
            "state": state,
            "shdist": row["shdist"],
            "nfhs_district_name": nfhs_dist_raw,
            "matched_census_district": matched_raw,
            "census_distid": matched_distid,
            "census_source_year": census_entry["source_year"],
            "match_score": score,
            "match_source": "fuzzy",
        }

        # Apply manual override if this (state, shdist, name) was hand-reviewed
        override_key = (state, row["shdist"], nfhs_dist_raw)
        if override_key in MANUAL_OVERRIDES:
            record["census_distid"] = MANUAL_OVERRIDES[override_key]
            record["match_score"] = 100
            record["match_source"] = "manual_override"

        results.append(record)

        if record["match_score"] < MATCH_THRESHOLD:
            flags.append({**record, "reason": "low fuzzy match score"})
 
    return pd.DataFrame(results), pd.DataFrame(flags)
 
 
def main():
    print("Loading raw files...")
    nfhs1 = pd.ExcelFile(f"{RAW_DIR}/NFHS-1_DistrictCodes.xlsx").parse("District Codes")
    nfhs1 = nfhs1.rename(columns={"hv024.1": "state_name"})
 
    nfhs2 = pd.ExcelFile(f"{RAW_DIR}/NFHS-2_DistrictCodes.xlsx").parse("District Codes")
    nfhs2 = nfhs2.rename(columns={"hv024.1": "state_name"})
 
    census91 = pd.ExcelFile(f"{RAW_DIR}/dist_list_81_91.xlsx").parse("1991 Districts")
    census81 = pd.ExcelFile(f"{RAW_DIR}/dist_list_81_91.xlsx").parse("1981 Districts")
 
    print("Building Census lookup (1991 primary, 1981 fallback)...")
    census_lookup = build_census_lookup(census91, census81)
 
    print("Matching NFHS-1 districts...")
    xwalk1, flags1 = match_round(nfhs1, census_lookup, "NFHS-1")
 
    print("Matching NFHS-2 districts...")
    xwalk2, flags2 = match_round(nfhs2, census_lookup, "NFHS-2")

    # Global district key -- census_distid alone is only unique WITHIN a state
    # (same issue as the original shdist), so build a state-qualified key here
    # before export. This is the join key to use when building the panel.
    for xwalk in (xwalk1, xwalk2):
        xwalk["district_uid"] = xwalk["state"] + "_" + xwalk["census_distid"].astype("Int64").astype(str)
        xwalk.loc[xwalk["census_distid"].isna(), "district_uid"] = None

    xwalk1.to_csv(f"{OUT_DIR}/nfhs1_district_crosswalk.csv", index=False)
    xwalk2.to_csv(f"{OUT_DIR}/nfhs2_district_crosswalk.csv", index=False)
 
    all_flags = pd.concat([flags1, flags2], ignore_index=True)
    all_flags.to_csv(f"{OUT_DIR}/crosswalk_review_flags.csv", index=False)
 
    print("\n--- SUMMARY ---")
    print(f"NFHS-1: {len(xwalk1)} state-district rows, "
          f"{(xwalk1['match_score'] < MATCH_THRESHOLD).sum()} flagged for review "
          f"(avg score {xwalk1['match_score'].mean():.1f})")
    print(f"NFHS-2: {len(xwalk2)} state-district rows, "
          f"{(xwalk2['match_score'] < MATCH_THRESHOLD).sum()} flagged for review "
          f"(avg score {xwalk2['match_score'].mean():.1f})")
    print(f"\nOutputs written to {OUT_DIR}/")
    print("Review crosswalk_review_flags.csv before trusting the full panel merge.")
 
 
if __name__ == "__main__":
    main()

Loading raw files...
Building Census lookup (1991 primary, 1981 fallback)...
Matching NFHS-1 districts...
Matching NFHS-2 districts...

--- SUMMARY ---
NFHS-1: 393 state-district rows, 5 flagged for review (avg score 98.8)
NFHS-2: 440 state-district rows, 6 flagged for review (avg score 98.9)

Outputs written to /Users/eknoorsandhu/DHS_India_Research/data/processed/district_crosswalks/
Review crosswalk_review_flags.csv before trusting the full panel merge.


In [98]:
x1 = pd.read_csv(OUT_DIR / "nfhs1_district_crosswalk.csv")
print(x1['district_uid'].isna().sum(), "rows with null district_uid")  # should equal NE row count
print(x1['district_uid'].duplicated().sum(), "duplicate district_uid values")  # should be 0

3 rows with null district_uid
5 duplicate district_uid values


In [99]:
NE_STATES = ["arunachal pradesh", "manipur", "meghalaya", "mizoram", "nagaland"]

for round_label, path in [
    ("NFHS-1", str(OUT_DIR / "nfhs1_district_crosswalk.csv")),
    ("NFHS-2", str(OUT_DIR / "nfhs2_district_crosswalk.csv")),
]:
    df = pd.read_csv(path)
    df["_state_norm"] = df["state"].astype(str).str.strip().str.lower()
    ne_rows = df[df["_state_norm"].isin(NE_STATES)]
    still_flagged = ne_rows[
        (ne_rows["match_score"] < 90) |
        (ne_rows["census_distid"].isna()) |
        (ne_rows["census_distid"] == 0)
    ]
    print(f"--- {round_label} ---")
    print(f"NE rows total: {len(ne_rows)}, still flagged: {len(still_flagged)}")
    print(ne_rows["match_source"].value_counts())
    print()

--- NFHS-1 ---
NE rows total: 29, still flagged: 0
match_source
manual_override    29
Name: count, dtype: int64

--- NFHS-2 ---
NE rows total: 34, still flagged: 0
match_source
manual_override    34
Name: count, dtype: int64



In [100]:
x2 = pd.read_csv(OUT_DIR / "nfhs2_district_crosswalk.csv")
dupe_uids = x2[x2["district_uid"].duplicated(keep=False)].sort_values("district_uid")
print(dupe_uids[["state", "shdist", "nfhs_district_name", "matched_census_district", "census_distid", "district_uid", "match_score", "match_source"]].to_string(index=False))

  state  shdist nfhs_district_name matched_census_district  census_distid district_uid  match_score match_source
    goa       1          North Goa                     Goa              0        goa_0    90.000000        fuzzy
    goa       2          South Goa                     Goa              0        goa_0    90.000000        fuzzy
 sikkim       1     NORTH DISTRICT                  Sikkim              0     sikkim_0    30.000000        fuzzy
 sikkim       2      EAST DISTRICT                  Sikkim              0     sikkim_0    31.578947        fuzzy
 sikkim       3     SOUTH DISTRICT                  Sikkim              0     sikkim_0    30.000000        fuzzy
 sikkim       4      WEST DISTRICT                  Sikkim              0     sikkim_0    31.578947        fuzzy
tripura       1       WEST TRIPURA                 Tripura              0    tripura_0    90.000000        fuzzy
tripura       2      NORTH TRIPURA                 Tripura              0    tripura_0    90.000

In [101]:
# ---------------------------------------------------------------------------
# NFHS-4/5 district crosswalk: restricted to the DHS-recommended stable
# baseline (codes present unchanged in both rounds; codes >= 801 in NFHS-5
# are post-2016 split-created districts, excluded per DHS user forum
# guidance -- see https://userforum.dhsprogram.com/index.php?t=msg&goto=24551).
# No attempt is made to reconstruct split parents; no official concordance
# for that exists (confirmed: neither MoHFW nor ICF/DHS has released one).
# ---------------------------------------------------------------------------

MICRO_DIR = REPO_ROOT / "data" / "raw" / "DHS_microdata"

_, meta = pyreadstat.read_dta(
    MICRO_DIR / "NFHS4_2015-16_IndividualRecode" / "IAIR74FL.DTA",
    metadataonly=True,
)
_, meta5 = pyreadstat.read_dta(
    MICRO_DIR / "NFHS5_2019-21_IndividualRecode" / "IAIR7EFL.DTA",
    metadataonly=True,
)

labels4 = meta.variable_value_labels["sdistri"]   # NFHS-4: code -> district name
labels5 = meta5.variable_value_labels["sdist"]    # NFHS-5: code -> district name

codes4 = set(labels4.keys())
codes5 = set(labels5.keys())

stable_codes = codes4 & codes5          # present unchanged in both rounds
dropped_codes = codes4 - codes5         # NFHS-4 parents that were split
new_codes = codes5 - codes4             # NFHS-5 children (post-split, code >= 801)

print(f"Stable districts: {len(stable_codes)}")
print(f"Dropped (NFHS-4 only, split parents): {len(dropped_codes)}")
print(f"New (NFHS-5 only, split children): {len(new_codes)}")

# --- Build the crosswalk table ---
nfhs4_5_crosswalk = pd.DataFrame([
    {
        "district_code": code,
        "district_name_nfhs4": labels4[code],
        "district_name_nfhs5": labels5[code],
        "status": "stable",
    }
    for code in sorted(stable_codes)
])

# Sanity check: names should match (or be trivially close) for every stable
# code, since the code itself is the join key. Flag any mismatch for review
# rather than silently trusting the code alignment.
name_mismatches = nfhs4_5_crosswalk[
    nfhs4_5_crosswalk["district_name_nfhs4"] != nfhs4_5_crosswalk["district_name_nfhs5"]
]
print(f"\nStable codes with a name mismatch between rounds: {len(name_mismatches)}")
if len(name_mismatches):
    print(name_mismatches.to_string(index=False))

# --- Excluded districts, for transparency / paper appendix ---
excluded_districts = pd.DataFrame([
    {"district_code": code, "district_name": labels4[code], "excluded_reason": "split_between_nfhs4_and_nfhs5"}
    for code in sorted(dropped_codes)
])

print(f"\nExcluded parent districts (NFHS-4 codes with no NFHS-5 match): {len(excluded_districts)}")
print(excluded_districts.to_string(index=False))

# --- Save outputs ---
nfhs4_5_crosswalk.to_csv(OUT_DIR / "nfhs4_nfhs5_stable_district_crosswalk.csv", index=False)
excluded_districts.to_csv(OUT_DIR / "nfhs4_nfhs5_excluded_split_districts.csv", index=False)

print(f"\nSaved {len(nfhs4_5_crosswalk)} stable districts to nfhs4_nfhs5_stable_district_crosswalk.csv")
print(f"Saved {len(excluded_districts)} excluded districts to nfhs4_nfhs5_excluded_split_districts.csv")


Stable districts: 575
Dropped (NFHS-4 only, split parents): 65
New (NFHS-5 only, split children): 132

Stable codes with a name mismatch between rounds: 12
 district_code                  district_name_nfhs4       district_name_nfhs5 status
             3                                  leh               leh(ladakh) stable
            25                      lahul and spiti             lahul & spiti stable
            39                              sangrur shahid bhagat singh nagar stable
            53            shahid bhagat singh nagar                   sangrur stable
           184                      siddharth nagar            siddharthnagar stable
           248                            papumpare                papum pare stable
           272 senapati (excluding 3 sub-divisions)                  senapati stable
           369                  saraikela kharsawan       saraikela-kharsawan stable
           400                       korea (koriya)                    koriya s

In [102]:
CODE_SWAP_NFHS5_TO_NFHS4 = {39: 53, 53: 39}

nfhs4_5_crosswalk["district_code_nfhs5"] = nfhs4_5_crosswalk["district_code"].apply(
    lambda c: CODE_SWAP_NFHS5_TO_NFHS4.get(c, c)
)

# Rebuild district_name_nfhs5 using the swap-corrected code, then confirm
# the mismatch count drops from 12 to the expected 10 (cosmetic-only).
nfhs4_5_crosswalk["district_name_nfhs5"] = nfhs4_5_crosswalk["district_code_nfhs5"].map(labels5)

name_mismatches = nfhs4_5_crosswalk[
    nfhs4_5_crosswalk["district_name_nfhs4"].str.strip().str.lower()
    != nfhs4_5_crosswalk["district_name_nfhs5"].str.strip().str.lower()
]
print(f"Remaining mismatches: {len(name_mismatches)}")  # expect 10, all cosmetic
print(name_mismatches.to_string(index=False))

nfhs4_5_crosswalk.to_csv(OUT_DIR / "nfhs4_nfhs5_stable_district_crosswalk.csv", index=False)
print("Re-saved with swap fix applied.")

Remaining mismatches: 10
 district_code                  district_name_nfhs4     district_name_nfhs5 status  district_code_nfhs5
             3                                  leh             leh(ladakh) stable                    3
            25                      lahul and spiti           lahul & spiti stable                   25
           184                      siddharth nagar          siddharthnagar stable                  184
           248                            papumpare              papum pare stable                  248
           272 senapati (excluding 3 sub-divisions)                senapati stable                  272
           369                  saraikela kharsawan     saraikela-kharsawan stable                  369
           400                       korea (koriya)                  koriya stable                  400
           407                            kabirdham              kabeerdham stable                  407
           469                         

In [103]:
def build_gps_district_reference(dbf_path, code_col="DHSREGCO", name_col="DHSREGNA"):
    gdf = gpd.read_file(dbf_path)
    return dict(zip(gdf[code_col], gdf[name_col]))

gps4_ref = build_gps_district_reference(MICRO_DIR / "NFHS4_2015-16_GPS" / "IAGE71FL.dbf")
gps5_ref = build_gps_district_reference(MICRO_DIR / "NFHS5_2019-21_GPS" / "IAGE7AFL.dbf")

pd.DataFrame(gps4_ref.items(), columns=["district_code", "district_name"]).to_csv(
    OUT_DIR / "nfhs4_gps_district_reference.csv", index=False)
pd.DataFrame(gps5_ref.items(), columns=["district_code", "district_name"]).to_csv(
    OUT_DIR / "nfhs5_gps_district_reference.csv", index=False)

In [104]:
print(len(gps4_ref), "codes in NFHS-4 GPS reference")
print(len(gps5_ref), "codes in NFHS-5 GPS reference")
print((OUT_DIR / "nfhs4_gps_district_reference.csv").exists())
print((OUT_DIR / "nfhs5_gps_district_reference.csv").exists())

640 codes in NFHS-4 GPS reference
707 codes in NFHS-5 GPS reference
True
True
